![image_1780319930905.png](./image_1780319930905.png "image_1780319930905.png")

![image_1780319946344.png](./image_1780319946344.png "image_1780319946344.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

# Initialize Spark session
spark = SparkSession.builder.appName("MatchesDataFrame").getOrCreate()

# Matches dataset
matches_data = [
    (1, "Arsenal", "Chelsea", 3, 1),
    (2, "Chelsea", "Liverpool", 2, 2),
    (3, "Liverpool", "Arsenal", 1, 0),
    (4, "Arsenal", "Liverpool", 2, 1),
    (5, "Chelsea", "Arsenal", 0, 2),
    (6, "Liverpool", "Chelsea", 3, 0),
]
matches_columns = ["match_id", "home_team", "away_team", "home_score", "away_score"]

matches_df = spark.createDataFrame(matches_data, matches_columns)

display(matches_df)

In [0]:
team_a_df = (
    matches_df.groupBy("home_team")
    .agg(
        f.sum("home_score").alias("goals_scored"),
        f.sum("away_score").alias("away_score"),
    )
    .select(
        f.col("home_team").alias("team"), f.col("goals_scored"), f.col("away_score")
    )
)
team_b_df = (
    matches_df.groupBy("away_team")
    .agg(
        f.sum("away_score").alias("goals_scored"),
        f.sum("home_score").alias("away_score"),
    )
    .select(
        f.col("away_team").alias("team"), f.col("goals_scored"), f.col("away_score")
    )
)
result_df = (
    (
        team_a_df.join(team_b_df, on="team").select(
            (team_a_df.team).alias("team"),
            (team_a_df.goals_scored + team_b_df.goals_scored).alias("goals_scored"),
            (team_a_df.away_score + team_b_df.away_score).alias("goals_conceded"),
        )
    )
    .withColumn("goal_difference", f.col("goals_scored") - f.col("goals_conceded"))
    .orderBy(f.col("goal_difference").desc(), f.col("team").asc())
)

display(result_df)

In [0]:
result_df = (
    matches_df
    .select(
        f.col("home_team").alias("team"),
        f.col("home_score").alias("goals_scored"),
        f.col("away_score").alias("goals_conceded")
    )
    .union(
        matches_df.select(
            f.col("away_team").alias("team"),
            f.col("away_score").alias("goals_scored"),
            f.col("home_score").alias("goals_conceded")
        )
    )
    .groupBy("team")
    .agg(
        f.sum("goals_scored").alias("goals_scored"),
        f.sum("goals_conceded").alias("goals_conceded")
    )
    .withColumn("goal_difference", f.col("goals_scored") - f.col("goals_conceded"))
    .orderBy(f.col("goal_difference").desc(), f.col("team").asc())
)

display(result_df)